# Predictive Maintenance — XGBoost Model

**Goal:** Train a binary classifier to predict machine failure.
<br>
**Input:** Sensor readings + machine type
<br>
**Target:** Machine failure (binary)

## What I expect
- Tool wear and torque will be the most important features
- The temp_diff engineered feature may improve performance
- Raw accuracy will be high but misleading — I'll focus on recall and F1
- SHAP will confirm or contradict my EDA intuitions

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report,
                            confusion_matrix,
                            ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder 
import xgboost as xgb
import shap

RANDOM_STATE = 42

In [2]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv"
df = pd.read_csv(url)

In [3]:
df = df.drop(columns=['UDI', 'Product ID'])

In [4]:
le = LabelEncoder()
df['Type'] = le.fit_transform(df['Type'])
print(le.classes_)

['H' 'L' 'M']


In [6]:
df['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
print(df['temp_diff'].describe())

count    10000.000000
mean        10.000630
std          1.001094
min          7.600000
25%          9.300000
50%          9.800000
75%         11.000000
max         12.100000
Name: temp_diff, dtype: float64


In [8]:
drop_cols=['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
X = df.drop(columns=drop_cols)
y = df['Machine failure']

print(f"Features: {X.columns.tolist()}")
print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

Features: ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'temp_diff']
Feature matrix shape: (10000, 7)
Target distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2, # 80% train 20% test
    random_state = RANDOM_STATE,
    stratify = y # preserve the 3.4% failure rate in both splits
)

print(f"Train size: {X_train.shape[0]} rows")
print(f"Test size: {X_test.shape[0]} rows")
print(f"Failure rate in train: {y_train.mean():.3f}")
print(f"Failure rate in test: {y_test.mean():.3f}")

Train size: 8000 rows
Test size: 2000 rows
Failure rate in train: 0.034
Failure rate in test: 0.034
